# FvCB

The FvCB model, also known as the Farquhar, von Caemmerer, and Berry model, is a widely used theoretical framework in plant biology, first conceptualized in 1980. It provides a simplistic view of C3 photosynthesis and is named after its primary contributors: Graham D. Farquhar, Susanne von Caemmerer, and Joseph A. Berry. 


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def min_solve(a: float, b: float, c: float) -> float:
    """Minimum of quadratic solutions of a * x**2 + b * x + c"""
    right = math.sqrt((b**2) - (4 * (a * c))) / (2 * a)
    return min(-b + right, -b - right)

## Core (min-A)

At it's core the model focuses on the carbon fixation stage of photosynthesis and describes **carbon assimilation (A)** by dividing it into three categories: 

- the RuBisCO limited rate (Ac)
- the electron transport limited rate (Aj)
- and the triphosphate limited rate (Ap)

The carbon assimilation rate is then ultimately the minimum of these three processes

**Note that is is not recommended to use this variant**, as predictions for net carbon assimilation (`An`) disagree with measurements when the partial pressure of CO2 is lower than the CO2 compensation point (`C < Gamma*`) (see [Lochocki 2025](https://doi.org/10.1093/insilicoplants/diaf014))

In [ ]:
def carbon_assimilation_min_a(
    ac: float,  # rubisco-limited rate
    aj: float,  # electron-transport-limited rate
    ap: float,  # triphosphate-limited rate
) -> float:
    return min(ac, aj, ap)


fig, ax = plt.subplots(
    figsize=(4, 3),
    layout="constrained",
)

limits = pd.DataFrame(
    {
        "ac": np.linspace(0, 50, 10),
        "aj": np.linspace(10, 30, 10),
        "ap": np.linspace(20, 25, 10),
    }
)

limits.plot(ax=ax, alpha=0.5)
ax.plot(
    limits.apply(lambda x: carbon_assimilation_min_a(*x), axis=1),
    label="A",
)
ax.legend()
ax.set(
    title="FvCB - min A",
    xlabel="Probably intercellular carbon",
    ylabel="Carbon fixation rate",
)
plt.show()

## min-W

- 
- Substracts losses by non-photorespiratory CO2

In [ ]:
def net_carbon_assimilation(
    wc: float,  # rubisco-limited rate
    wj: float,  # electron-transport-limited rate
    wp: float,  # triphosphate-limited rate
    c: float,  #  partical pressure of CO2 in the vicinity of rubisco
    cp: float = 38.6,  # CO2 compensation point [Γ*] / mubar
    rl: float = 1,  # rate of non-photorespiratory CO2 release in the light / μmol m⁻² s⁻¹
) -> float:
    vc = min(wc, wj, wp)
    return vc * (1 - cp / c) - rl


fig, ax = plt.subplots(
    figsize=(4, 3),
    layout="constrained",
)

limits = pd.DataFrame(
    {
        "wc": np.linspace(0, 50, 10),
        "wj": np.linspace(10, 30, 10),
        "wp": np.linspace(20, 25, 10),
    }
)
c = 100

limits.plot(ax=ax, alpha=0.5)
ax.plot(
    limits.apply(lambda x: net_carbon_assimilation(*x, c=c), axis=1),
    label="A",
)
ax.legend()
ax.set(
    title="FvCB - min W",
    xlabel="Probably intercellular carbon",
    ylabel="Carbon fixation rate",
)
plt.show()

In reality, we can't directly measure any of these three rates, so we have to find ways to approximate them.  



## Version 1

Assumptions:

- mesophyll resistance is 0



In [ ]:
def ac_v1(
    ci: float,  # intercellular CO2 concentration
    o: float,  # intercellular O2 concentration
    compensation_point: float,  # CO2 compensation point
    vcmax: float,  # maximal rubisco rate
    kc: float,  # rubisco km for CO2
    ko: float,  # rubisco km for O2
    rd: float,  # respiration rate
) -> float:
    return ((ci - compensation_point) * vcmax) / (ci + kc * (1 + o / ko)) - rd


def aj_v1(
    ci: float,
    compensation_point: float,  # CO2 compensation point
    j: float,  # electron transport rate
    rd: float,  # respiration rate
) -> float:
    return ((ci - compensation_point) * j) / (4 * ci + 8 * compensation_point) - rd


def ap_v1(
    tp: float,
    rd: float,  # respiration rate
) -> float:
    return 3 * tp - rd

In [ ]:
ci = np.linspace(0, 700, 100)
o: float = 210  # mbar
j: float = 124  # µmol m⁻² s⁻¹
tp: float = 15  # µmol m⁻² s⁻¹
vcmax = 80  # µmol m⁻² s⁻¹
kc = 259  # µbar
ko = 179  # mbar
compensation_point = 38.6  # µbar
rd = 1  # µmol m⁻² s⁻¹

limits = pd.DataFrame(
    {
        c: {
            "ac": ac_v1(c, o, compensation_point, vcmax, kc, ko, rd),
            "aj": aj_v1(c, compensation_point, j, rd),
            "ap": ap_v1(tp, rd),
        }
        for c in ci
    }
).T


fig, ax = plt.subplots(
    figsize=(4, 3),
    layout="constrained",
)

limits.plot(ax=ax, alpha=0.5)
ax.plot(
    limits.apply(lambda x: net_carbon_assimilation(*x), axis=1),
    label="A",
)
ax.legend()
ax.set(
    title="FvCB",
    xlabel="Probably intercellular carbon",
    ylabel="Carbon fixation rate",
)
plt.show()

## With mesophyll resistance

In [ ]:
def ac(
    ci: float,  # intercellular CO2 concentration
    o: float,  # intercellular O2 concentration
    compensation_point: float,  # CO2 compensation point (gamma*)
    vcmax: float,  # maximal rubisco rate
    kc: float,  # rubisco km for CO2
    ko: float,  # rubisco km for O2
    rd: float,  # respiration rate
    rm: float,  # mesophyll resistance
) -> float:
    return min_solve(
        a=1,
        b=-((ci + kc * (1 + o / ko)) / rm + vcmax - rd),
        c=(vcmax * (ci - compensation_point) - rd * (ci + kc * (1 + o / ko))) / rm,
    )

## 